### Comparing all LLMs response 

In [3]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import json
from  IPython.display  import Markdown

In [4]:
# Always remember to do this!
load_dotenv(override=True)

True

In [ ]:
# Print the key prefixes to help with any debugging

# openai_api_key = os.getenv('OPENAI_API_KEY')
# anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
# google_api_key = os.getenv('GOOGLE_API_KEY')
# deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')


# instead of above key will use github access token to access  models from github marketplace like openai , deepseek , groq

# model = "xai/grok-3-mini" # grok model 


github_token = os.getenv("GITHUB_TOKEN")   # will use for models openai , grok , deepseek 
github_base_url = os.getenv("GITHUB_BASE_URL")


google_api_key = os.getenv("GOOGLE_API_KEY")  #  for gemini 
groq_api_key = os.getenv("GROQ_API_KEY")  # for groq model

if github_token:
    print(f"github token is present  and begins {github_token[:8]}")
else:
    print("github token  not set")
    
if github_base_url:
    print(f"github base url exist and is   {github_base_url}")
else:
    print("github base url not exits ")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")


if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

In [6]:
request = "Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence. "
request += "Answer only with the question, no explanation."
messages = [{"role": "user", "content": request}]

In [ ]:
messages

In [ ]:
# getting  from openai model
openai = OpenAI(base_url=github_base_url, api_key=github_token)
response = openai.chat.completions.create(
    model="openai/gpt-4o-mini",
    messages=messages,
)
question = response.choices[0].message.content
print(question)


In [9]:

competitors = []
answers = []
messages = [{"role": "user", "content": question}]

In [ ]:


# Replace the model with gpt-4.1-mini if you'd prefer not to wait 1-2 mins

model_name = "gpt-4o-mini"

response = openai.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
# using gemini model 
gemini = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")

# model_name = "gemini-2.5-flash"
model_name = "gemini-2.5-flash-lite"

response = gemini.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
model_name = "deepseek/DeepSeek-V3-0324"

response = openai.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
model_name = "deepseek/DeepSeek-R1"

response = openai.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
# Updated with the latest Open Source model from OpenAI

groq = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")
model_name = "openai/gpt-oss-120b"

response = groq.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)


In [ ]:
!ollama pull qwen3:0.6b  # pulling the latest version of qwen3:0.6b model from ollama registry to run locally

In [ ]:
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
model_name = "qwen3:0.6b"

response = ollama.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)


In [ ]:
# So where are we? (all competitors and there answers )

print(competitors)
print(answers)

In [ ]:
# It's nice to know how to use "zip"
for competitor, answer in zip(competitors, answers):
    print(f"Competitor: {competitor}\n\n{answer}")

In [18]:
# Let's bring this together - note the use of "enumerate"

together = ""
for index, answer in enumerate(answers):
    together += f"# Response from competitor {index+1}\n\n"
    together += answer + "\n\n"

In [ ]:
print(together)

In [20]:
judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""


In [ ]:
#  get the judge's decision using gemini model , get the index of the best answer and print it out ( we passed index as model and their reposne so that judge can evaluate and rank them without any bias )
gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key=google_api_key)
model_name = "gemini-2.5-flash-lite"  # using gemini for judging 
response = gemini.chat.completions.create(model=model_name, messages=[{"role": "user", "content": judge}])
results = response.choices[0].message.content
print(results)

In [ ]:
# OK let's turn this into results!
# getting models names with their ranks and print them out

results_dict = json.loads(results)
ranks = results_dict["results"]
for index, result in enumerate(ranks):
    competitor = competitors[int(result)-1]
    print(f"Rank {index+1}: {competitor}")

## Exercise 
Using another design pattern 